# T5



In [2]:
import numpy as np
from scipy.stats import norm

np.random.seed(42)


In [3]:
theta_true = 2.0
n = 100
beta = 0.95
alpha = 1 - beta

x = np.random.uniform(theta_true, 2 * theta_true, size=n)

x_mean = x.mean()
x_max = x.max()

theta_mm = (2 / 3) * x_mean
theta_mle = x_max / 2
theta_mle_corr = ((n + 1) / (2 * n + 1)) * x_max

theta_mm, theta_mle, theta_mle_corr


(1.9602409911709453, 1.9868869366005173, 1.996771946235346)

## Точный доверительный интервал

Используем статистику

$$
Y = \frac{X_{(n)}}{\theta} - 1,
$$

для которой

$$
F_Y(y) = y^n, \quad y \in [0,1].
$$

Тогда центральный точный интервал уровня доверия $\beta$:


In [4]:
a = ((1 - beta) / 2) ** (1 / n)
b = ((1 + beta) / 2) ** (1 / n)

ci_exact = (
    x_max / (1 + b),
    x_max / (1 + a)
)

ci_exact


(1.9871384547089106, 2.0235297134456385)

## Асимптотический доверительный интервал

Для ОММ-оценки

$$
\hat\theta_{\text{ММ}} = \frac{2}{3}\bar X,
$$

имеем

$$
\sqrt{n}(\hat\theta_{\text{ММ}} - \theta) \Rightarrow N\left(0, \frac{\theta^2}{27}\right).
$$

После подстановки оценки вместо неизвестного параметра получаем:


In [5]:
z = norm.ppf((1 + beta) / 2)

se_asym = theta_mm / np.sqrt(27 * n)
ci_asym = (
    theta_mm - z * se_asym,
    theta_mm + z * se_asym
)

ci_asym


(1.8863016331389328, 2.0341803492029578)

## Bootstrap-интервал

Берём исправленную ОМП-оценку

$$
\hat\theta^{*} = \frac{n+1}{2n+1} X_{(n)}.
$$

Дальше строим percentile bootstrap.


In [9]:
B = 1000
theta_boot = []

for _ in range(B):
    sample = np.random.choice(x, size=n, replace=True)
    theta_b = (n + 1) / (2*n + 1) * np.max(sample)
    theta_boot.append(theta_b)

theta_boot = np.array(theta_boot)

alpha = 0.05
lower = np.quantile(theta_boot, alpha / 2)
upper = np.quantile(theta_boot, 1 - alpha / 2)
ci_boot =(lower, upper)

lower, upper

(1.9754112969207016, 1.996771946235346)

## Сравнение интервалов

По выборке были построены:

- точный доверительный интервал для параметра $\theta$;
- асимптотический доверительный интервал для параметра $\theta$;
- bootstrap-доверительный интервал для параметра $\theta$.

Точный интервал построен по точному распределению статистики $X_{(n)}$.

Асимптотический интервал построен для оценки метода моментов, так как модель
$U[\theta, 2\theta]$ нерегулярна и стандартную асимптотическую нормальность ОМП
здесь использовать нельзя.

Bootstrap-интервал является численным приближением, построенным по эмпирическому
распределению исправленной ОМП-оценки.

In [10]:
print("theta_true =", round(theta_true, 6))
print("theta_mm =", round(theta_mm, 6))
print("theta_mle =", round(theta_mle, 6))
print("theta_mle_corr =", round(theta_mle_corr, 6))
print()

print("exact CI =", tuple(round(v, 6) for v in ci_exact))
print("asymptotic CI =", tuple(round(v, 6) for v in ci_asym))
print("bootstrap CI =", tuple(round(v, 6) for v in ci_boot))

theta_true = 2.0
theta_mm = 1.960241
theta_mle = 1.986887
theta_mle_corr = 1.996772

exact CI = (1.987138, 2.02353)
asymptotic CI = (1.886302, 2.03418)
bootstrap CI = (1.975411, 1.996772)
